# Tutorial: Persistent Homology of Two Point Clouds with GUDHI

This notebook introduces a basic workflow in **topological data analysis (TDA)** using the Python library **GUDHI**.

We will compare two synthetic point clouds:

1. A noisy circle
2. Two separated Gaussian blobs

The noisy circle should produce a clear 1-dimensional topological feature: a loop. The two-blob dataset should mainly illustrate 0-dimensional behavior: connected components merging as the scale grows.

We will use a **Vietoris--Rips filtration**, compute persistent homology, and visualize persistence diagrams, barcodes, feature lifetimes, and Rips graphs at selected distance scales.

## 1. Install dependencies

Run the following cell if you do not already have the required packages installed.

In [ ]:
# Uncomment and run if needed:
# %pip install gudhi numpy matplotlib scikit-learn

## 2. Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gudhi as gd

from sklearn.datasets import make_circles, make_blobs

np.random.seed(42)

## 3. Create two point clouds

The first point cloud is sampled from a noisy circle. The second consists of two separated Gaussian blobs.

In [ ]:
# Point cloud 1: noisy circle
circle_points, _ = make_circles(
    n_samples=120,
    factor=0.98,
    noise=0.04,
    random_state=42
)

# Point cloud 2: two separated blobs
blob_points, _ = make_blobs(
    n_samples=120,
    centers=[[-1.5, 0], [1.5, 0]],
    cluster_std=0.25,
    random_state=42
)

## 4. Visualize the point clouds

Before computing topology, always inspect the data geometrically.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].scatter(circle_points[:, 0], circle_points[:, 1], s=20)
axes[0].set_title("Point Cloud A: Noisy Circle")
axes[0].set_aspect("equal")

axes[1].scatter(blob_points[:, 0], blob_points[:, 1], s=20)
axes[1].set_title("Point Cloud B: Two Blobs")
axes[1].set_aspect("equal")

plt.tight_layout()
plt.show()

### Interpretation

The circle data has an empty region in the middle. We expect persistent homology to detect this as a 1-dimensional hole.

The blob data has two separated clusters. We expect persistent homology to show many connected components at very small scales, then two main components, and eventually one connected component after the clusters merge.

## 5. Helper function for persistent homology

The function below builds a Vietoris--Rips complex from a point cloud, converts it to a simplex tree, and computes persistent homology.

In [ ]:
def compute_rips_persistence(points, max_edge_length=1.0, max_dimension=2):
    """
    Compute persistent homology of a point cloud using a Vietoris-Rips filtration.

    Parameters
    ----------
    points : np.ndarray
        Array of shape (n_points, n_dimensions).
    max_edge_length : float
        Maximum distance for adding edges to the Rips complex.
    max_dimension : int
        Maximum simplex dimension to construct.

    Returns
    -------
    simplex_tree : gudhi.SimplexTree
        The simplex tree representing the filtered complex.
    persistence : list
        Persistence pairs returned by GUDHI.
    """
    rips_complex = gd.RipsComplex(
        points=points,
        max_edge_length=max_edge_length
    )

    simplex_tree = rips_complex.create_simplex_tree(
        max_dimension=max_dimension
    )

    persistence = simplex_tree.persistence()

    return simplex_tree, persistence

For 2D point clouds where we want to detect loops, `max_dimension=2` is usually enough. Edges create loops, while triangles can fill those loops and mark their deaths in the filtration.

## 6. Compute persistence for both point clouds

In [ ]:
circle_st, circle_persistence = compute_rips_persistence(
    circle_points,
    max_edge_length=0.8,
    max_dimension=2
)

blob_st, blob_persistence = compute_rips_persistence(
    blob_points,
    max_edge_length=2.5,
    max_dimension=2
)

print("Circle point cloud")
print("Number of simplices:", circle_st.num_simplices())
print("Number of persistence pairs:", len(circle_persistence))

print("\nTwo-blob point cloud")
print("Number of simplices:", blob_st.num_simplices())
print("Number of persistence pairs:", len(blob_persistence))

## 7. Plot persistence diagrams

A persistence diagram plots birth time on the horizontal axis and death time on the vertical axis. Points far from the diagonal represent features that persist over a longer range of scales.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

gd.plot_persistence_diagram(circle_persistence, axes=axes[0])
axes[0].set_title("Persistence Diagram: Noisy Circle")

gd.plot_persistence_diagram(blob_persistence, axes=axes[1])
axes[1].set_title("Persistence Diagram: Two Blobs")

plt.tight_layout()
plt.show()

### Interpretation

For the noisy circle, look for a long-lived H1 point. This corresponds to the central loop.

For the two blobs, H0 points represent connected components. Most components die quickly as nearby points connect. A longer-lived H0 feature can reflect the separation between the two major clusters.

## 8. Plot persistence barcodes

A persistence barcode represents each topological feature as a horizontal interval. Long bars are usually more meaningful than short bars.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

gd.plot_persistence_barcode(circle_persistence, axes=axes[0])
axes[0].set_title("Persistence Barcode: Noisy Circle")

gd.plot_persistence_barcode(blob_persistence, axes=axes[1])
axes[1].set_title("Persistence Barcode: Two Blobs")

plt.tight_layout()
plt.show()

## 9. Visualize the Rips graph at selected scales

A useful teaching visualization is the graph obtained by connecting points whose pairwise distance is at most a chosen value $\epsilon$.

This graph is the 1-skeleton of the Vietoris--Rips complex.

In [ ]:
def plot_rips_graph(points, epsilon, ax, title):
    n = len(points)

    ax.scatter(points[:, 0], points[:, 1], s=20)

    for i in range(n):
        for j in range(i + 1, n):
            distance = np.linalg.norm(points[i] - points[j])

            if distance <= epsilon:
                ax.plot(
                    [points[i, 0], points[j, 0]],
                    [points[i, 1], points[j, 1]],
                    linewidth=0.5,
                    alpha=0.4
                )

    ax.set_title(title)
    ax.set_aspect("equal")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8))

circle_epsilons = [0.10, 0.20, 0.35]
blob_epsilons = [0.15, 0.40, 1.50]

for ax, eps in zip(axes[0], circle_epsilons):
    plot_rips_graph(
        circle_points,
        epsilon=eps,
        ax=ax,
        title=f"Circle, epsilon={eps}"
    )

for ax, eps in zip(axes[1], blob_epsilons):
    plot_rips_graph(
        blob_points,
        epsilon=eps,
        ax=ax,
        title=f"Blobs, epsilon={eps}"
    )

plt.tight_layout()
plt.show()

### Interpretation of the Rips graph sequence

For the circle:

- At small $\epsilon$, many points are isolated or only weakly connected.
- At medium $\epsilon$, the circle becomes connected while preserving a central hole.
- At larger $\epsilon$, edges and triangles eventually fill the hole.

For the blobs:

- At small $\epsilon$, only local neighborhoods form.
- At medium $\epsilon$, each blob becomes internally connected.
- At larger $\epsilon$, the two blobs connect to each other.

## 10. Discussion questions

1. Why does the circle point cloud have a stronger H1 signal than the blob point cloud?
2. What happens if the circle becomes noisier?
3. What happens if the two blobs are moved closer together?
4. Why do short persistence bars often correspond to noise?
5. Why do we need triangles, not just edges, to determine when a loop dies?

## 11. Key takeaway

Persistent homology tracks topological features across many distance scales. In this example, the noisy circle has a meaningful loop, visible as a long-lived H1 feature. The two-blob cloud is better described by its connected-component behavior in H0.

The basic GUDHI workflow is:

1. Build a filtered complex.
2. Compute persistence.
3. Visualize diagrams, barcodes, and lifetimes.
4. Interpret long-lived features as candidate signal and short-lived features as likely noise.